In [1]:
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

class LEVIRCDDataset(Dataset):
    def __init__(self, root, split="train", img_size=256, augment=False):
        self.img_size  = img_size
        self.crop_size = int(img_size * 0.875)  # 항상 img_size보다 작음
        self.augment   = augment        

        super().__init__()
        self.root = Path(root)
        self.split = split
        self.img_size = img_size
        self.augment = augment

        list_file = self.root / "list" / f"{split}.txt"
        with open(list_file, "r") as f:
            self.filenames = [line.strip() for line in f if line.strip()]

        self.dir_a     = self.root / split / "A"
        self.dir_b     = self.root / split / "B"
        self.dir_label = self.root / split / "label"

        self.img_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std =[0.229, 0.224, 0.225]),
        ])
        self.mask_transform = transforms.Compose([
            transforms.Resize((img_size, img_size),
                              interpolation=transforms.InterpolationMode.NEAREST),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx: int) -> dict:
        fname = self.filenames[idx]
        img_a = Image.open(self.dir_a     / fname).convert("RGB")
        img_b = Image.open(self.dir_b     / fname).convert("RGB")
        mask  = Image.open(self.dir_label / fname).convert("L")

        if self.augment:
            # ── 1. Random horizontal flip
            if torch.rand(1) > 0.5:
                img_a = transforms.functional.hflip(img_a)
                img_b = transforms.functional.hflip(img_b)
                mask  = transforms.functional.hflip(mask)

            # ── 2. Random vertical flip
            if torch.rand(1) > 0.5:
                img_a = transforms.functional.vflip(img_a)
                img_b = transforms.functional.vflip(img_b)
                mask  = transforms.functional.vflip(mask)
                
        img_a = self.img_transform(img_a)   # Resize(img_size) + Normalize
        img_b = self.img_transform(img_b)
        image = torch.stack([img_a, img_b], dim=0)  # [2, 3, H, W]

        mask_t = self.mask_transform(mask)           # Resize(img_size, NEAREST)
        mask_t = (mask_t > 0.5).long().squeeze(0)   # [H, W]

        return {"image": image, "mask": mask_t, "filename": fname}


def get_dataloader(root, split, img_size=256, batch_size=4,
                   num_workers=0, shuffle=None):
    is_train = split == "train"
    dataset = LEVIRCDDataset(root=root, split=split,
                             img_size=img_size, augment=is_train)
    return DataLoader(dataset,
                      batch_size=batch_size,
                      shuffle=is_train if shuffle is None else shuffle,
                      num_workers=num_workers,
                      pin_memory=True,
                      drop_last=is_train)


In [2]:
import shutil
from datetime import datetime
import torch.nn.functional as F
import lightning as L
from lightning.pytorch.callbacks import (ModelCheckpoint, EarlyStopping, 
                                         LearningRateMonitor, Callback
)
from lightning.pytorch.loggers import TensorBoardLogger
from torchgeo.models import ChangeViT
from torchgeo.models.changevit import DetailCaptureModule

# ─────────────────────────────────────────────
# 1. Config
"""
steps_per_epoch = len(train_loader)           # 445
MAX_ITERS       = steps_per_epoch * CFG["max_epochs"]  # 17,800

steps_per_epoch = 7120 / 16 = 445 steps
total_iters     = 445 x 40  = 17,800 iters -> 이 상태에서는 poly schedule의 LR이 거의 줄어들지 않음

마지막 iter의 lr = 2e-4 x (1 - 17800/80000)^0.9
                 = 2e-4 x 0.777^0.9
                 ≈ 2e-4 x 0.802
                 ≈ 1.6e-4   ← 거의 안 줄어듦
"""

CFG = dict(
    data_root    = "D:/Project-Dataset/LEVIR-CD-256",
    img_size     = 256,
    batch_size   = 16,
    num_workers  = 0,
    max_epochs   = 180,
    lr           = 2e-4,
    weight_decay = 1e-4,
    backbone     = 'vit_small_patch16_dinov3.lvd1689m',   # timm 
    # backbone     = "vit_small_patch14_dinov2.lvd142m",  # timm model
    # backbone     = 'facebook/dinov3-vits16-pretrain-lvd1689m',  # hugging face
    log_dir      = "logs/changevit",
    ckpt_base    = "checkpoints/changevit",   # 실험별 서브폴더 여기 아래 생성됨
)

# ─────────────────────────────────────────────
# 실험 ID: 훈련 시작 시각 → 항상 새 폴더 생성
RUN_ID   = datetime.now().strftime("%Y%m%d_%H%M%S")
CKPT_DIR = Path(CFG["ckpt_base"]) / RUN_ID
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# ─────────────────────────────────────────────
# 2. Best-checkpoint → best.pth 복사 콜백
class SaveBestPth(Callback):
    """
    ModelCheckpoint가 best 파일을 갱신할 때마다
    같은 폴더에 best.pth 로 복사해 둔다.
    """
    def __init__(self, ckpt_callback: ModelCheckpoint):
        self.ckpt_callback = ckpt_callback

    def on_validation_end(self, trainer, pl_module):
        best = self.ckpt_callback.best_model_path
        if best and Path(best).exists():
            dst = Path(best).parent / "best.pth"
            shutil.copy2(best, dst)


# ─────────────────────────────────────────────
# 3. LightningModel
class LightningModel(L.LightningModule):
    def __init__(self, cfg: dict, max_iters: int):
        super().__init__()
        self.save_hyperparameters({**cfg, "max_iters": max_iters})
        self.max_iters  = max_iters
        self._optimizer = None          # setup()에서 단 한 번 생성

        self.model = ChangeViT(
            backbone    = cfg["backbone"],
            img_size    = cfg["img_size"],
            in_channels = 3,
            num_classes = 1,
            pretrained  = True,
        )
        # detailcapture backbone 변경 
        new_detail = DetailCaptureModule(
            in_channels = 6,
            # backbone    = "resnet18.a1_in1k",  
            backbone    = "resnet34.a1_in1k",  
            pretrained  = True,
        )
        self.model.detail_capture = new_detail
        
    # ── LR schedule (논문 poly + warm-up) ────────────
    def adjust_learning_rate(self, cur_iter: int) -> float:
        """
        논문 수식: lr = base_lr x (1 - cur_iter / max_iter)^0.9
        epoch=0 초기 200 iter: linear warm-up
            lr = base_lr x [0.9 x (iter+1)/200 + 0.1]
        cur_iter: 전체 step 기준 (epoch x steps_per_epoch + batch_idx)
        """
        base_lr = self.hparams["lr"]
        # warm-up 구간 (첫 번째 epoch 초반 200 iter)
        if cur_iter < 200:
            lr = base_lr * (0.9 * (cur_iter + 1) / 200 + 0.1)
        else:
            lr = base_lr * (1 - cur_iter / self.max_iters) ** 0.9
        # eta_min 하한 (논문에는 명시 없으나 안전장치)
        lr = max(lr, 1e-6)
        for pg in self._optimizer.param_groups:
            pg["lr"] = lr
        return lr

    # ── Loss ─────────────────────────────────────────
    def BCEDiceLoss(self, logits, targets):
        targets_f = targets.unsqueeze(1).float()
        bce   = F.binary_cross_entropy_with_logits(logits, targets_f)
        probs = torch.sigmoid(logits.float())
        inter = (probs * targets_f).sum()
        eps   = 1e-5
        dice  = (2 * inter + eps) / (probs.sum() + targets_f.sum() + eps)
        return bce + 1 - dice

    # ── Metrics: F1 / IoU / OA ───────────────────────
    @torch.no_grad()
    def _metrics(self, logits, targets):
        """
        Returns: f1, iou, oa  (pixel-level binary)
        OA = Overall Accuracy = (TP + TN) / N
        """
        preds = (torch.sigmoid(logits) > 0.5).long().squeeze(1)   # [B,H,W]
        tp = ((preds == 1) & (targets == 1)).sum().float()
        tn = ((preds == 0) & (targets == 0)).sum().float()
        fp = ((preds == 1) & (targets == 0)).sum().float()
        fn = ((preds == 0) & (targets == 1)).sum().float()

        precision = tp / (tp + fp + 1e-5)
        recall    = tp / (tp + fn + 1e-5)
        f1  = 2 * precision * recall / (precision + recall + 1e-5)
        iou = tp / (tp + fp + fn + 1e-5)
        oa  = (tp + tn) / (tp + tn + fp + fn + 1e-5)
        return f1, iou, oa

    # ── Gradient norm ─────────────────────────────────
    def log_gradient_norms(self):
        total = torch.tensor(0.0, device=self.device)
        for p in self.parameters():
            if p.grad is not None:
                total += p.grad.detach().norm(2) ** 2
        return total.sqrt()

    # def on_train_epoch_end(self, trainer, pl_module):
    #     if self.trainer.current_epoch == 0:
    #         self.trainer.save_checkpoint("first_epoch.ckpt")


    # ── Steps ─────────────────────────────────────────
    def training_step(self, batch, batch_idx):
        # poly LR 갱신
        steps_per_epoch = self.trainer.num_training_batches
        cur_iter = self.current_epoch * steps_per_epoch + batch_idx
        lr = self.adjust_learning_rate(cur_iter)

        image, mask = batch["image"], batch["mask"]
        logits = self.model(image)
        loss   = self.BCEDiceLoss(logits, mask)

        f1, iou, oa = self._metrics(logits, mask)
        self.log("train/loss", loss, prog_bar=True,  on_step=True,  on_epoch=True)
        self.log("train/f1",   f1,   prog_bar=False, on_step=False, on_epoch=True)
        self.log("train/iou",  iou,  prog_bar=False, on_step=False, on_epoch=True)
        self.log("train/oa",   oa,   prog_bar=False, on_step=False, on_epoch=True)
        self.log("train/lr",   lr,   prog_bar=True,  on_step=True,  on_epoch=False)
        return loss

    def on_after_backward(self):
        gnorm = self.log_gradient_norms()
        # print(f"[hook 호출됨] grad_norm={gnorm:.4f}")  # 확인
        self.log("train/grad_norm", gnorm, prog_bar=False, on_step=True, on_epoch=False)

    def validation_step(self, batch, batch_idx):
        image, mask = batch["image"], batch["mask"]
        logits = self.model(image)
        loss   = self.BCEDiceLoss(logits, mask)

        f1, iou, oa = self._metrics(logits, mask)
        self.log("val/loss", loss, prog_bar=True,  on_step=False, on_epoch=True)
        self.log("val/f1",   f1,  prog_bar=True,  on_step=False, on_epoch=True)
        self.log("val/iou",  iou, prog_bar=True,  on_step=False, on_epoch=True)
        self.log("val/oa",   oa,  prog_bar=True,  on_step=False, on_epoch=True)

    def test_step(self, batch, batch_idx):
        image, mask = batch["image"], batch["mask"]
        logits = self.model(image)
        loss   = self.BCEDiceLoss(logits, mask)
        f1, iou, oa = self._metrics(logits, mask)
        self.log("test/loss", loss, on_epoch=True)
        self.log("test/f1",   f1,  on_epoch=True, prog_bar=True)
        self.log("test/iou",  iou, on_epoch=True)
        self.log("test/oa",   oa,  on_epoch=True)

    # ── Optimizer (스케줄러 없음 — 수동 poly 사용) ────
    def configure_optimizers(self):
        self._optimizer = torch.optim.AdamW(
            self.parameters(),
            lr           = self.hparams["lr"],
            betas        = (0.9, 0.99),          # 논문 설정
            weight_decay = self.hparams["weight_decay"],
            eps=1e-08  # github ChangeViT/main.py
        )
        return self._optimizer   # lr_scheduler 반환 안 함

In [3]:
test_loader = get_dataloader(
    root        = CFG["data_root"],
    split       = "test",
    img_size    = CFG["img_size"],
    batch_size  = CFG["batch_size"],
    num_workers = CFG["num_workers"],
)

#----------------   BACBONE CAHNGE  -----------------------
# val/f1 = 0.8073 with backbone = "resnet34.a1_in1k" 
#BEST_CKPT = "./checkpoints/changevit/20260419_022016/best.pth"
# val/f1 = 0.8134 with backbone = "resnet18.a1_in1k"  
BEST_CKPT = "./checkpoints/changevit/20260420-vit16_resnet34_best.pth"
#-------------------------------------------------------------
# test 전용 trainer — callbacks, logger 최소화
test_trainer = L.Trainer(
    accelerator = "gpu",
    devices     = 1,
    precision   = "16-mixed",
    logger      = False,   # TensorBoard 기록 불필요하면 False
    enable_progress_bar = True,
)

# model은 ckpt_path에서 가중치를 자동으로 덮어쓰므로 재생성할 필요 없음
# 단, LightningModel 인스턴스는 필요 (아키텍처 정의용)
test_model = LightningModel(CFG, max_iters=10)

results = test_trainer.test(test_model, test_loader, ckpt_path=BEST_CKPT)
print(results)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
You are using a CUDA device ('NVIDIA GeForce RTX 3060 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
Restoring states from the checkpoint path at ./checkpoints/changev

Output()

d:\Project-TorchGeo\.venv_py312\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
d:\Project-TorchGeo\.venv_py312\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


d:\Project-TorchGeo\.venv_py312\Lib\site-packages\lightning\pytorch\utilities\data.py:79: Trying to infer the 
`batch_size` from an ambiguous collection. The batch size we found is 16. To avoid any miscalculations, use 
`self.log(..., batch_size=batch_size)`.

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          test/f1          │    0.7702240347862244     │
│         test/iou          │    0.6703689098358154     │
│         test/loss         │    0.3157828152179718     │
│          test/oa          │    0.9863207936286926     │
└───────────────────────────┴───────────────────────────┘

[{'test/loss': 0.3157828152179718, 'test/f1': 0.7702240347862244, 'test/iou': 0.6703689098358154, 'test/oa': 0.9863207936286926}]
